# CommitBench filtering — Python only (local CSV files)

This notebook reads the CommitBench CSV files that you manually downloaded into `data/raw/`.
It does **not** download the dataset from Hugging Face.

## Expected input files
- `data/raw/train.csv`
- `data/raw/val.csv` (or `validation.csv`)
- `data/raw/test.csv`

The notebook also checks the exact filenames and prints what it finds. It expects each CSV to contain a diff column and a commit-message column (usually `diff` and `message`).

## Outputs
- `data/processed/train.jsonl`
- `data/processed/validation.jsonl`
- `data/processed/test.jsonl`
- `data/processed/plots/original_dataset_overview.png`
- `data/processed/plots/cleaned_dataset_overview.png`
- `commitbench-python-filtered.zip`

Each JSONL record contains `diff` and `message`.

## Filtering steps
- Keeps Python-only changes based on changed-file extensions, with language metadata as a fallback.
- Keeps Conventional Commit messages and removes low-content messages.
- Removes generated/lock files, truncates long diffs, and limits files per diff.
- Deduplicates within each split and removes placeholder tokens from labels.
- Caps `fix` examples at 12,000 in the training split only.

## Run in Kiro / VS Code
1. Keep the notebook in your `OffHand` project folder.
2. Confirm your manually downloaded CSVs are in `OffHand/data/raw/`.
3. Select the project Python environment as the notebook kernel.
4. Run all cells from top to bottom.

Set `LIMIT_PER_SPLIT = 2000` in the configuration cell for a small test run. Set it to `None` to process every row. The limit counts **input rows scanned per split**, so it also speeds up the original-data graphs.


In [ ]:
# Install dependencies into the active notebook kernel if needed.
# This uses local CSVs; it does not download CommitBench itself.
%pip install -q datasets tqdm matplotlib


In [ ]:
import hashlib
import json
import re
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm.auto import tqdm


In [ ]:
# ---------------------------------------------------------------------------
# Configuration — local Kiro / VS Code
# ---------------------------------------------------------------------------

# Preferred path from your project. If the notebook is moved, the relative
# fallback is checked as well.
PROJECT_DIR = Path(r"C:\Users\ASUSD\Desktop\Programing\seniorProject\OffHand")
CANDIDATE_RAW_DIRS = [PROJECT_DIR / "data" / "raw", Path.cwd() / "data" / "raw"]
RAW_DIR = next((p for p in CANDIDATE_RAW_DIRS if p.is_dir()), CANDIDATE_RAW_DIRS[0])

# Output paths are relative to the project folder when it exists.
OUTPUT_ROOT = PROJECT_DIR if PROJECT_DIR.is_dir() else Path.cwd()
OUT_DIR = OUTPUT_ROOT / "data" / "processed"
PLOT_DIR = OUT_DIR / "plots"

# Set to e.g. 2_000 for a quick test; use None for all input rows.
# This limit applies to rows SCANNED, not rows successfully kept.
LIMIT_PER_SPLIT = None

RANDOM_SEED = 42
TARGET_LANGUAGE = "python"
MAX_PER_TYPE = {"fix": 12_000}  # training split only; set {} to disable
FIGURE_DPI = 140

print(f"Project/output root: {OUTPUT_ROOT.resolve()}")
print(f"Raw data directory:  {RAW_DIR.resolve()}")
print(f"Output directory:    {OUT_DIR.resolve()}")
print(f"Row limit per split: {LIMIT_PER_SPLIT}")


In [ ]:
# ---------------------------------------------------------------------------
# Filtering helpers (mirrors src/commit_model/diff_utils.py — kept inline so
# the notebook is self-contained and runs on Kaggle / Colab without the package)
# ---------------------------------------------------------------------------

CONVENTIONAL_PREFIX = re.compile(
    r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)(\([\w./-]+\))?!?:\s+\S"
)

# Regex that matches the type prefix only (used for per-type counting)
TYPE_RE = re.compile(r"^(\w+)(\(.+?\))?!?:")

LOW_CONTENT_STOPLIST = {
    "fix", "wip", "update", "updates", "minor", "minor changes", "misc",
    "cleanup", "small fix", "typo", "changes", "stuff", "test", "tmp",
    "asdf", "oops", "fixup",
}

GENERATED_FILE_PATTERNS = [
    re.compile(p) for p in [
        r"package-lock\.json$", r"yarn\.lock$", r"pnpm-lock\.yaml$",
        r"Cargo\.lock$", r"Gemfile\.lock$", r"poetry\.lock$", r"uv\.lock$",
        r"\.min\.(js|css)$", r"\.map$", r"(?:^|/)dist/", r"(?:^|/)build/", r"(?:^|/)vendor/",
        r"\.svg$", r"\.snap$",
    ]
]

MAX_DIFF_CHARS = 4000
MAX_FILES_PER_DIFF = 6

# Placeholders injected by CommitBench into anonymised messages.
# Strip them from training data so the model never learns to generate them.
# (At inference time, postprocess.py also strips them — this is the training-side fix.)
_PLACEHOLDER_RE = re.compile(
    r"<(I|URL|EMAIL|HASH)>"
    r"|\(#<I>\)"
    r"|\b[0-9a-f]{7,40}\b"  # bare hex hashes that escaped the <HASH> anonymisation
    r"|https?://\S+"         # bare URLs that escaped the <URL> anonymisation
)


def is_generated_path(path: str) -> bool:
    return any(p.search(path) for p in GENERATED_FILE_PATTERNS)


def message_is_low_content(message: str) -> bool:
    first_line = message.strip().splitlines()[0] if message.strip() else ""
    has_prefix = CONVENTIONAL_PREFIX.match(first_line)
    subject = first_line.split(":", 1)[-1].strip() if has_prefix else first_line
    normalized = subject.lower().strip(" .!")
    return len(normalized) < 8 or normalized in LOW_CONTENT_STOPLIST


def split_diff_by_file(diff: str):
    """Split a unified diff into (path, hunk_text) chunks."""
    chunks = re.split(r"(?=^diff --git )", diff, flags=re.MULTILINE)
    files = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        match = re.search(r"^diff --git a/(\S+) b/(\S+)", chunk)
        path = match.group(2) if match else "unknown"
        files.append((path, chunk))
    return files


def truncate_diff(diff: str):
    """Drop generated/lockfile paths, cap file count, and cap total length."""
    files = split_diff_by_file(diff)
    kept = [(p, c) for p, c in files if not is_generated_path(p)]
    if not kept:
        return None
    kept = kept[:MAX_FILES_PER_DIFF]
    pieces = []
    for path, chunk in kept:
        share = max(MAX_DIFF_CHARS // max(len(kept), 1), 200)
        pieces.append(chunk[:share])
    return "".join(pieces)[:MAX_DIFF_CHARS]


def clean_training_message(message: str) -> str:
    """Strip CommitBench placeholder tokens from a training message.

    Does NOT apply the lowercase-first-word cosmetic from postprocess.py —
    that is inference-time only and should not alter training labels.
    Returns an empty string if nothing meaningful remains after cleaning.
    """
    cleaned = _PLACEHOLDER_RE.sub("", message)
    # Collapse multiple spaces and trim
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned).strip()
    # Collapse dangling punctuation left behind: 'fix: (#)' -> 'fix:'
    cleaned = re.sub(r":\s*[()#\[\]]+\s*$", "", cleaned).strip()
    return cleaned


def extract_type(message: str):
    """Return the commit type prefix string, or None."""
    m = TYPE_RE.match(message.strip())
    return m.group(1) if m else None


# ---------------------------------------------------------------------------
# Programming-language filtering
# ---------------------------------------------------------------------------

_LANGUAGE_ALIASES = {
    "py": "python",
    "python": "python",
    "js": "javascript",
    "javascript": "javascript",
    "ts": "typescript",
    "typescript": "typescript",
    "java": "java",
    "go": "go",
    "php": "php",
    "rb": "ruby",
    "ruby": "ruby",
    "c++": "cpp",
    "cpp": "cpp",
    "py3": "python",
}


def _normalise_language(value: str) -> str:
    value = str(value).strip().lower()
    return _LANGUAGE_ALIASES.get(value, value)


def _languages_from_metadata(row: dict):
    """Return normalized language metadata, or None when no usable field exists."""
    for key in ("diff_languages", "languages", "language"):
        if key not in row or row[key] is None:
            continue
        value = row[key]
        if isinstance(value, (list, tuple, set)):
            raw_values = list(value)
        else:
            raw_values = re.split(r"[,;|]", str(value))
        languages = {
            _normalise_language(v)
            for v in raw_values
            if str(v).strip() and str(v).strip().lower() not in {"none", "nan", "unknown"}
        }
        if languages:
            return languages
    return None


def _languages_from_diff_paths(diff: str):
    """Infer languages from changed file extensions; unknown extensions are rejected."""
    paths = re.findall(
        r"^diff --git a/(.*?) b/(.*?)$",
        diff,
        flags=re.MULTILINE,
    )
    if not paths:
        return None

    extension_to_language = {".py": "python"}
    languages = set()
    for old_path, new_path in paths:
        path = new_path or old_path
        suffix = Path(path).suffix.lower()
        language = extension_to_language.get(suffix)
        if language is None:
            return {"__non_python_or_unknown__"}
        languages.add(language)
    return languages or None


def is_target_language_example(row: dict, target_language: str = TARGET_LANGUAGE) -> bool:
    """Keep only diffs whose changed-file paths confirm the target language.

    File extensions take priority because the requirement is Python-only changed
    files. Metadata is used only when paths cannot be parsed from the diff.
    """
    diff = row.get("diff") or ""
    languages_from_paths = _languages_from_diff_paths(diff)
    if languages_from_paths is not None:
        return languages_from_paths == {target_language}

    languages_from_metadata = _languages_from_metadata(row)
    return languages_from_metadata == {target_language}


In [ ]:
# ---------------------------------------------------------------------------
# build_example — the central per-row filter function
# ---------------------------------------------------------------------------

def build_example(diff: str, message: str, seen_hashes: set, type_counts: dict, is_train: bool):
    """Return a filtered example dict, or None if the row should be dropped.

    Args:
        diff:        raw diff string from CommitBench
        message:     raw commit message from CommitBench
        seen_hashes: mutable set of (diff+message) hash strings — dedups within a split
        type_counts: mutable counter of kept examples per type — used for class rebalancing
        is_train:    True when processing the train split; class cap only applies then

    Returns:
        {"diff": str, "message": str} or None
    """
    message = message.strip()
    if not message:
        return None

    first_line = message.splitlines()[0]

    # --- Format check (must pass BEFORE cleaning to avoid accepting borderline messages) ---
    if not CONVENTIONAL_PREFIX.match(first_line):
        return None
    if message_is_low_content(message):
        return None

    # --- Diff quality check ---
    truncated = truncate_diff(diff)
    if truncated is None or len(truncated.strip()) < 20:
        return None

    # --- Placeholder cleaning (applied after format/quality gates) ---
    cleaned_message = clean_training_message(first_line)
    # If cleaning gutted the description (empty after the colon), reject the example
    if ":" not in cleaned_message:
        return None
    description_part = cleaned_message.split(":", 1)[1].strip()
    if len(description_part) < 4:
        return None
    # Re-validate format after cleaning
    if not CONVENTIONAL_PREFIX.match(cleaned_message):
        return None

    # --- Deduplication (within-split, by diff+message hash) ---
    h = hashlib.sha256((truncated + cleaned_message).encode()).hexdigest()[:16]
    if h in seen_hashes:
        return None
    seen_hashes.add(h)

    # --- Class rebalancing (train split only) ---
    commit_type = extract_type(cleaned_message)
    if is_train and commit_type and commit_type in MAX_PER_TYPE:
        if type_counts.get(commit_type, 0) >= MAX_PER_TYPE[commit_type]:
            return None

    # All checks passed — update counters and return the example
    if commit_type:
        type_counts[commit_type] = type_counts.get(commit_type, 0) + 1

    return {"diff": truncated, "message": cleaned_message}

In [ ]:
# ---------------------------------------------------------------------------
# Load the manually downloaded local CommitBench CSV files
# ---------------------------------------------------------------------------

if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f"Raw data directory does not exist: {RAW_DIR}\n"
        "Create data/raw and put the downloaded CommitBench CSV files there."
    )

csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        f"No .csv files found in {RAW_DIR}. Found files: "
        f"{[p.name for p in RAW_DIR.iterdir()]}"
    )

print("CSV files found:")
for file in csv_files:
    print(f"  {file.name} ({file.stat().st_size / (1024**2):.1f} MiB)")

# Match common filename variants without requiring exact capitalization.
def find_split_file(aliases):
    for file in csv_files:
        stem = file.stem.lower().replace("-", "_").replace(" ", "_")
        if stem in aliases:
            return file
    for file in csv_files:
        stem = file.stem.lower().replace("-", "_").replace(" ", "_")
        if any(alias in stem for alias in aliases):
            return file
    return None

train_file = find_split_file({"train", "training", "train_set"})
val_file = find_split_file({"val", "valid", "validation", "dev", "val_set"})
test_file = find_split_file({"test", "testing", "test_set"})

if train_file is None:
    raise FileNotFoundError(
        "Could not identify the training CSV. Expected a filename like train.csv. "
        f"CSV files found: {[p.name for p in csv_files]}"
    )

# Map local files to canonical Hugging Face split names.
data_files = {"train": str(train_file)}
if val_file is not None:
    data_files["validation"] = str(val_file)
if test_file is not None:
    data_files["test"] = str(test_file)

print("\nLoading local CSV files with Hugging Face Datasets (no network download):")
for split_name, file_path in data_files.items():
    print(f"  {split_name}: {file_path}")

ds = load_dataset("csv", data_files=data_files)

# Normalize common column-name variants to the names expected by the filter.
for split_name in list(ds.keys()):
    columns = list(ds[split_name].column_names)
    lower_to_actual = {column.lower().strip(): column for column in columns}

    diff_candidates = ("diff", "patch", "code_diff", "diff_text", "changes")
    message_candidates = ("message", "commit_message", "commit_msg", "subject", "commitmessage")

    diff_col = next((lower_to_actual[name] for name in diff_candidates if name in lower_to_actual), None)
    message_col = next((lower_to_actual[name] for name in message_candidates if name in lower_to_actual), None)

    if diff_col is None or message_col is None:
        raise ValueError(
            f"Split '{split_name}' has columns {columns}, but the notebook needs a diff/patch column "
            "and a commit-message column. Please inspect the CSV headers and update the aliases."
        )

    rename_map = {}
    if diff_col != "diff":
        rename_map[diff_col] = "diff"
    if message_col != "message":
        rename_map[message_col] = "message"
    if rename_map:
        ds[split_name] = ds[split_name].rename_columns(rename_map)

print("\nLoaded local dataset:")
for split_name, split in ds.items():
    print(f"  {split_name}: {len(split):,} rows; columns={split.column_names}")

print("\nSample training row:")
print({key: (str(value)[:250] + "...") if isinstance(value, str) and len(value) > 250 else value
       for key, value in ds["train"][0].items()})


In [ ]:
# ---------------------------------------------------------------------------
# Overview plots — original dataset BEFORE filtering
# ---------------------------------------------------------------------------

def _safe_first_line(message):
    return (message or "").strip().splitlines()[0] if (message or "").strip() else ""


def _language_labels_for_plot(row):
    # Prefer parsed changed-file extensions, then fall back to dataset metadata.
    by_path = _languages_from_diff_paths(row.get("diff") or "")
    langs = by_path if by_path is not None else _languages_from_metadata(row)
    return sorted(langs) if langs else ["unknown"]


original_split_sizes = {}
original_types = Counter()
original_message_lengths = []
original_languages = Counter()

for split_name, split in ds.items():
    rows_to_analyze = split
    if LIMIT_PER_SPLIT is not None:
        rows_to_analyze = split.select(range(min(len(split), LIMIT_PER_SPLIT)))
    original_split_sizes[split_name] = len(rows_to_analyze)
    for row in tqdm(rows_to_analyze, desc=f"Analyze original {split_name}", total=len(rows_to_analyze)):
        message = row.get("message") or ""
        first_line = _safe_first_line(message)
        commit_type = extract_type(first_line)
        if commit_type:
            original_types[commit_type] += 1
        if message.strip():
            original_message_lengths.append(len(first_line))
        for lang in _language_labels_for_plot(row):
            original_languages[lang] += 1

PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0, 0].bar(list(original_split_sizes.keys()), list(original_split_sizes.values()))
axes[0, 0].set_title("Original rows per split")
axes[0, 0].set_ylabel("Rows")
axes[0, 0].tick_params(axis="x", rotation=20)

top_languages = original_languages.most_common(12)
if top_languages:
    axes[0, 1].bar([x[0] for x in top_languages], [x[1] for x in top_languages])
    axes[0, 1].set_title("Original detected languages (top 12)")
    axes[0, 1].tick_params(axis="x", rotation=35)
else:
    axes[0, 1].set_title("Original detected languages (no metadata/paths found)")

top_types = original_types.most_common(12)
if top_types:
    axes[1, 0].bar([x[0] for x in top_types], [x[1] for x in top_types])
    axes[1, 0].set_title("Original Conventional Commit types (top 12)")
    axes[1, 0].tick_params(axis="x", rotation=35)
else:
    axes[1, 0].set_title("Original commit types (none detected)")

if original_message_lengths:
    axes[1, 1].hist(original_message_lengths, bins=35)
axes[1, 1].set_title("Original first-line message length")
axes[1, 1].set_xlabel("Characters")
axes[1, 1].set_ylabel("Examples")

fig.suptitle("CommitBench — before filtering", fontsize=15)
fig.tight_layout()
original_plot_path = PLOT_DIR / "original_dataset_overview.png"
fig.savefig(original_plot_path, dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print(f"Saved original-data plots to: {original_plot_path.resolve()}")


In [ ]:
# ---------------------------------------------------------------------------
# Filter & write Python-only examples
# ---------------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Map Hugging Face split names to the requested output filenames.
OUTPUT_SPLIT_NAMES = {
    "train": "train",
    "val": "validation",
    "validation": "validation",
    "test": "test",
}

for split_name in ds.keys():
    if split_name not in OUTPUT_SPLIT_NAMES:
        print(f"Skipping unexpected split '{split_name}'.")
        continue

    split = ds[split_name]
    if LIMIT_PER_SPLIT is not None:
        split = split.select(range(min(len(split), LIMIT_PER_SPLIT)))
    output_name = OUTPUT_SPLIT_NAMES[split_name]
    out_path = OUT_DIR / f"{output_name}.jsonl"
    is_train = split_name == "train"

    # Reset per-split state so deduplication does not cross splits.
    seen_hashes: set = set()
    type_counts: dict = {}

    kept = 0
    seen = 0
    python_rows = 0
    language_removed = 0
    placeholder_cleaned = 0

    with out_path.open("w", encoding="utf-8") as f:
        for row in tqdm(split, desc=f"{split_name} -> {output_name}", total=len(split)):
            seen += 1

            # Apply the language filter before other quality filters.
            if not is_target_language_example(row, TARGET_LANGUAGE):
                language_removed += 1
                continue
            python_rows += 1

            diff = row.get("diff") or ""
            message = (row.get("message") or "").strip()
            if not diff or not message:
                continue

            # Track whether the original subject contained placeholders.
            has_placeholder = bool(
                re.search(r"<(I|URL|EMAIL|HASH)>", message.splitlines()[0])
            )

            example = build_example(
                diff=diff,
                message=message,
                seen_hashes=seen_hashes,
                type_counts=type_counts,
                is_train=is_train,
            )
            if example is None:
                continue

            if has_placeholder:
                placeholder_cleaned += 1

            f.write(json.dumps(example, ensure_ascii=False) + "\n")
            kept += 1

    print(
        f"\n{split_name}: kept {kept:,} examples from {seen:,} rows "
        f"({kept / max(seen, 1):.1%}) -> {out_path}"
    )
    print(f"  Python-only rows: {python_rows:,}")
    print(f"  Removed by language filter: {language_removed:,}")
    print(f"  Placeholder-containing messages included: {placeholder_cleaned:,}")
    if is_train and MAX_PER_TYPE:
        print(f"  Class rebalancing active (MAX_PER_TYPE={MAX_PER_TYPE})")
    print("  Per-type breakdown:")
    for t, n in sorted(type_counts.items(), key=lambda x: -x[1]):
        pct = n / max(kept, 1)
        capped = is_train and t in MAX_PER_TYPE and n >= MAX_PER_TYPE[t]
        cap_note = f" [CAPPED at {MAX_PER_TYPE[t]}]" if capped else ""
        print(f"    {t:<12} {n:>6}  ({pct:.1%}){cap_note}")

print("\nFiltering complete. Expected outputs:")
for name in ("train.jsonl", "validation.jsonl", "test.jsonl"):
    print(f"  {OUT_DIR / name}")


In [ ]:
# ---------------------------------------------------------------------------
# Sanity checks + overview plots — cleaned dataset AFTER filtering
# ---------------------------------------------------------------------------

available_output_files = {
    split_name: OUT_DIR / f"{split_name}.jsonl"
    for split_name in ("train", "validation", "test")
}
cleaned_split_sizes = {}
cleaned_types = Counter()
cleaned_message_lengths = []
total_checked = 0

for split_name, file_path in available_output_files.items():
    if not file_path.exists():
        print(f"Not present (source split may not exist): {file_path}")
        continue

    split_count = 0
    with file_path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            assert set(row.keys()) == {"diff", "message"}, (
                f"{file_path}:{line_number} has unexpected keys: {row.keys()}"
            )
            assert isinstance(row["diff"], str) and isinstance(row["message"], str)
            assert row["diff"].strip(), f"{file_path}:{line_number} has an empty diff"
            assert row["message"].strip(), f"{file_path}:{line_number} has an empty message"
            for token in ("<I>", "<URL>", "<EMAIL>", "<HASH>"):
                assert token not in row["message"], (
                    f"{file_path}:{line_number} still contains placeholder {token}"
                )

            # Verify the changed paths when the diff format exposes them.
            languages = _languages_from_diff_paths(row["diff"])
            if languages is not None:
                assert languages == {"python"}, (
                    f"{file_path}:{line_number} is not Python-only: {languages}"
                )

            commit_type = extract_type(row["message"])
            if commit_type:
                cleaned_types[commit_type] += 1
            cleaned_message_lengths.append(len(_safe_first_line(row["message"])))
            split_count += 1
            total_checked += 1

    cleaned_split_sizes[split_name] = split_count
    print(f"{split_name}: {split_count:,} cleaned examples")

assert cleaned_split_sizes.get("train", 0) > 0, (
    "train.jsonl is missing or empty. Check the filters and source split names."
)
print(f"\nTotal cleaned examples checked: {total_checked:,}")
print("Schema, non-empty fields, placeholder, and detectable language checks passed.")

# Show a few training examples for a quick manual review.
train_path = OUT_DIR / "train.jsonl"
with train_path.open("r", encoding="utf-8") as f:
    print("\nSample training examples:")
    for i, line in enumerate(f):
        if i >= 3:
            break
        row = json.loads(line)
        print(f"  [{i + 1}] MESSAGE: {row['message']}")
        print(f"      DIFF:    {row['diff'][:180]}...\n")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
if cleaned_split_sizes:
    axes[0].bar(list(cleaned_split_sizes.keys()), list(cleaned_split_sizes.values()))
axes[0].set_title("Cleaned rows per split")
axes[0].set_ylabel("Rows")
axes[0].tick_params(axis="x", rotation=20)

top_cleaned_types = cleaned_types.most_common(12)
if top_cleaned_types:
    axes[1].bar([x[0] for x in top_cleaned_types], [x[1] for x in top_cleaned_types])
axes[1].set_title("Cleaned Conventional Commit types (top 12)")
axes[1].tick_params(axis="x", rotation=35)

if cleaned_message_lengths:
    axes[2].hist(cleaned_message_lengths, bins=30)
axes[2].set_title("Cleaned first-line message length")
axes[2].set_xlabel("Characters")
axes[2].set_ylabel("Examples")

fig.suptitle("CommitBench — after filtering", fontsize=15)
fig.tight_layout()
cleaned_plot_path = PLOT_DIR / "cleaned_dataset_overview.png"
fig.savefig(cleaned_plot_path, dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print(f"Saved cleaned-data plots to: {cleaned_plot_path.resolve()}")


In [ ]:
# ---------------------------------------------------------------------------
# Optional: package processed JSONL files and plots into a ZIP file
# ---------------------------------------------------------------------------

zip_base = OUTPUT_ROOT / "commitbench-python-filtered"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUT_DIR)
print(f"Created ZIP: {Path(zip_path).resolve()}")
